# 02 — Build silver
Cleans bronze into 25 typed columns with a unique `flight_key`, then
MERGEs into `flights.silver.flights`. Safe to re-run.

In [0]:
from pyspark.sql import functions as F

df = spark.read.table("flights.bronze.flights_raw")

silver = df.select(
        F.concat_ws("|",
        F.col("Reporting_Airline"),
        F.col("Flight_Number_Reporting_Airline"),
        F.col("FlightDate"),
        F.col("CRSDepTime"),
        F.col("Origin")
    ).alias("flight_key"),
    F.to_date("FlightDate").alias("flight_date"),
    F.col("Reporting_Airline").alias("airline_code"),
    F.col("Flight_Number_Reporting_Airline").alias("flight_number"),
    F.col("Tail_Number").alias("tail_number"),
    F.col("Origin").alias("origin_airport"),
    F.col("Dest").alias("dest_airport"),
    F.col("CRSDepTime").alias("scheduled_dep_time"),
    F.col("DepTime").alias("actual_dep_time"),
    F.col("DepDelay").cast("double").alias("dep_delay_min"),
    F.col("ArrDelay").cast("double").alias("arr_delay_min"),
    (F.col("Cancelled").cast("double") == 1).alias("is_cancelled"),
    (F.col("Diverted").cast("double") == 1).alias("is_diverted"),
    F.col("Distance").cast("double").alias("distance_miles"),
    F.col("AirTime").cast("double").alias("air_time_min"),
    F.col("CRSArrTime").alias("scheduled_arr_time"),
    F.col("ArrTime").alias("actual_arr_time"),
    (F.col("DepDel15").cast("double") == 1).alias("dep_delayed_15"),
    (F.col("ArrDel15").cast("double") == 1).alias("arr_delayed_15"),
    F.col("CancellationCode").alias("cancellation_code"),
    F.col("ActualElapsedTime").cast("double").alias("elapsed_time_min"),
    F.col("CarrierDelay").cast("double").alias("carrier_delay_min"),
    F.col("WeatherDelay").cast("double").alias("weather_delay_min"),
    F.col("NASDelay").cast("double").alias("nas_delay_min"),
    F.col("SecurityDelay").cast("double").alias("security_delay_min"),
    F.col("LateAircraftDelay").cast("double").alias("late_aircraft_delay_min")
)

silver.write.mode("overwrite") \
    .option("overwriteSchema", "true") \
    .saveAsTable("flights.silver.flights")

spark.read.table("flights.silver.flights").count()

In [0]:
total = silver.count()
distinct_keys = silver.select("flight_key").distinct().count()

print(f"rows: {total}, distinct keys: {distinct_keys}")
assert total == distinct_keys, f"Duplicate keys found: {total - distinct_keys}"

# new — null rate
null_count = silver.filter(F.col("flight_date").isNull()).count()
null_rate = null_count / total
print(f"flight_date null rate: {null_rate:.2%}")
assert null_rate < 0.01, f"flight_date null rate {null_rate:.2%} exceeds 1%"

# new — reconciliation
bronze_count = spark.read.table("flights.bronze.flights_raw").count()
print(f"bronze: {bronze_count}, silver: {total}")
assert bronze_count == total, \
    f"Row count mismatch: bronze={bronze_count}, silver={total}"

In [0]:
# Quarantine — cancelled flights with departure times
quarantine = silver.filter(
    (F.col("is_cancelled") == True) & 
    (F.col("actual_dep_time").isNotNull())
)

quarantine_count = quarantine.count()
print(f"Quarantine rows: {quarantine_count}")

if quarantine_count > 0:
    quarantine.write.mode("overwrite") \
        .saveAsTable("flights.silver.quarantine_cancelled_with_departure")

In [0]:
from delta.tables import DeltaTable

table_name = "flights.silver.flights"

if spark.catalog.tableExists(table_name):
    target = DeltaTable.forName(spark, table_name)
    
    (target.alias("t")
        .merge(
            silver.alias("s"),
            "t.flight_key = s.flight_key"
        )
        .whenMatchedUpdateAll()
        .whenNotMatchedInsertAll()
        .execute())
    
    print("merged")
else:
    silver.write.saveAsTable(table_name)
    print("created")